# VerinBirds — Этап 4: CNN transfer learning (ResNet18)

**Запускать в Google Colab с GPU** (Runtime → Change runtime type → T4 GPU). Локально не обучаем.

Что делает ноутбук:
1. Забирает куриный датасет (тот же, что в EDA/baseline: 3 класса `fertile/infertile/dead`, сплит training/testing).
2. Дообучает ResNet18 (предобучен на ImageNet): backbone заморожен, учим голову + последний блок `layer4`, ранняя остановка по val.
3. Считает честные метрики на test: accuracy, per-class recall, confusion matrix.
4. Сохраняет веса `verinbirds_resnet18.pt` и `metrics.json`.

**Честная рамка:** высокая точность на курином test ОЖИДАЕМА и мало что значит —
классы в этом датасете разделимы по цвету/источнику (baseline дал 99.8%). Настоящая проверка —
перенос на перепелов (Этап 6, локально). Здесь наша цель — получить веса и убедиться, что CNN
обучается на структуре, а не сломан.

## Что тебе сделать (по шагам)
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Выполни ячейку 1 — проверка GPU (должно напечатать `cuda`).
3. **Ячейка «Данные»**: смонтируй Google Drive и положи туда архив датасета
   (`kaggle_archive` целиком или zip). Поправь путь `DRIVE_ARCHIVE` под свой.
4. Запусти все ячейки подряд (Runtime → Run all).
5. В конце скачаются `verinbirds_resnet18.pt` и `metrics.json`.
   Положи `.pt` в репозиторий в `ml/models/`, а **содержимое `metrics.json` сохрани** —
   сравни его с baseline (см. baseline.py).


In [ ]:
# Ячейка 1 — проверка окружения и GPU
import torch, torchvision, sys
print("python:", sys.version.split()[0])
print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cpu":
    print("!!! GPU не подключён. Runtime -> Change runtime type -> T4 GPU, потом Run all заново.")


## Данные

Два способа. **Рекомендую способ A (Google Drive)** — ты уже скачал датасет локально.

- **A. Google Drive:** заархивируй локально папку `data/chicken/kaggle_archive` в `kaggle_archive.zip`,
  загрузи её в свой Google Drive (например в корень `My Drive`), и укажи путь ниже.
- **B. Kaggle API:** если положишь `kaggle.json` — можно скачать прямо в Colab (код закомментирован).

Ноутбук ждёт итоговую структуру `/content/data/chicken/**` с файлами вида
`.../training/fertile.fertile1099....jpg` (класс закодирован в имени, сплит — папкой training/testing).

In [ ]:
# Ячейка «Данные» — способ A: Google Drive
from google.colab import drive
drive.mount('/content/drive')

import zipfile, os, glob, pathlib
# >>> ПОПРАВЬ ПУТЬ под свой файл в Drive:
DRIVE_ARCHIVE = '/content/drive/MyDrive/kaggle_archive.zip'
DEST = '/content/data/chicken'
os.makedirs(DEST, exist_ok=True)

if DRIVE_ARCHIVE.endswith('.zip'):
    with zipfile.ZipFile(DRIVE_ARCHIVE) as z:
        z.extractall(DEST)
else:  # если указал папку на Drive — просто копируем
    import shutil; shutil.copytree(DRIVE_ARCHIVE, DEST, dirs_exist_ok=True)

imgs = glob.glob(DEST + '/**/*.jpg', recursive=True) + glob.glob(DEST + '/**/*.png', recursive=True)
print("изображений найдено:", len(imgs))
assert len(imgs) > 3000, "Мало файлов — проверь путь DRIVE_ARCHIVE и структуру архива"

# --- Способ B (Kaggle), при желании раскомментируй:
# !pip -q install kaggle
# from google.colab import files; files.upload()   # выбрать kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d hlnkmb/chicken-egg-analysis-dataset -p /content/data/chicken --unzip


In [ ]:
# Ячейка 3 — конфиг + препроцессинг (letterbox как в ml/src/preprocess.py) + аугментации
import glob, random, numpy as np
from pathlib import Path
from PIL import Image, ImageOps
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
IMG_SIZE = 224
CLASSES = ["fertile", "infertile", "dead"]
CLS2IDX = {c: i for i, c in enumerate(CLASSES)}
IMAGENET_MEAN = (0.485, 0.456, 0.406); IMAGENET_STD = (0.229, 0.224, 0.225)

class LetterboxResize:
    "Вписать в квадрат с сохранением пропорций + чёрный паддинг (== preprocess.py)"
    def __init__(self, size=IMG_SIZE): self.size = size
    def __call__(self, im):
        im = ImageOps.exif_transpose(im).convert("RGB")
        w, h = im.size; s = self.size / max(w, h)
        nw, nh = max(1, round(w*s)), max(1, round(h*s))
        im = im.resize((nw, nh), Image.BILINEAR)
        canvas = Image.new("RGB", (self.size, self.size), (0, 0, 0))
        canvas.paste(im, ((self.size-nw)//2, (self.size-nh)//2))
        return canvas

# Аугментации — обоснованы под овоскопирование (Этап 2): flip, поворот, яркость/контраст. Без hue/shear.
train_tf = T.Compose([LetterboxResize(), T.RandomHorizontalFlip(),
                      T.RandomRotation(15, fill=0),
                      T.ColorJitter(brightness=0.2, contrast=0.2),
                      T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_tf  = T.Compose([LetterboxResize(), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

def cls_of(name):
    n = name.lower()
    return "infertile" if "infertile" in n else "fertile" if "fertile" in n else "dead" if "dead" in n else None

class EggDS(Dataset):
    def __init__(self, files, tf): self.files = files; self.tf = tf
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        p = self.files[i]
        with Image.open(p) as im: x = self.tf(im)
        return x, CLS2IDX[cls_of(Path(p).name)]


In [ ]:
# Ячейка 4 — сборка сплитов: train/val из training/, test из testing/
root = Path('/content/data/chicken')
train_files = [str(p) for p in root.rglob('*') if p.suffix.lower() in {'.jpg','.png','.jpeg'} and 'training' in str(p).lower() and cls_of(p.name)]
test_files  = [str(p) for p in root.rglob('*') if p.suffix.lower() in {'.jpg','.png','.jpeg'} and 'testing'  in str(p).lower() and cls_of(p.name)]

from sklearn.model_selection import train_test_split
y_tr = [cls_of(Path(p).name) for p in train_files]
tr_files, val_files = train_test_split(train_files, test_size=0.15, stratify=y_tr, random_state=SEED)

from collections import Counter
print("train:", len(tr_files), Counter(cls_of(Path(p).name) for p in tr_files))
print("val:  ", len(val_files), Counter(cls_of(Path(p).name) for p in val_files))
print("test: ", len(test_files), Counter(cls_of(Path(p).name) for p in test_files))

BATCH = 32
dl_train = DataLoader(EggDS(tr_files, train_tf), batch_size=BATCH, shuffle=True, num_workers=2)
dl_val   = DataLoader(EggDS(val_files, eval_tf), batch_size=BATCH, shuffle=False, num_workers=2)
dl_test  = DataLoader(EggDS(test_files, eval_tf), batch_size=BATCH, shuffle=False, num_workers=2)


In [ ]:
# Ячейка 5 — модель: ResNet18 (ImageNet), заморозка backbone, учим layer4 + голову
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
for p in model.parameters(): p.requires_grad = False          # заморозили всё
for p in model.layer4.parameters(): p.requires_grad = True     # разморозили последний блок
model.fc = nn.Linear(model.fc.in_features, len(CLASSES))       # новая 3-классовая голова
model = model.to(DEVICE)

params = [p for p in model.parameters() if p.requires_grad]
print("обучаемых тензоров:", len(params))
opt = torch.optim.Adam(params, lr=1e-4)
crit = nn.CrossEntropyLoss()


In [ ]:
# Ячейка 6 — обучение с ранней остановкой по val loss
import copy
def run_epoch(dl, train):
    model.train() if train else model.eval()
    tot, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if train: opt.zero_grad()
            out = model(x); loss = crit(out, y)
            if train: loss.backward(); opt.step()
            loss_sum += loss.item()*x.size(0); tot += x.size(0)
            correct += (out.argmax(1) == y).sum().item()
    return loss_sum/tot, correct/tot

MAX_EPOCHS, PATIENCE = 15, 3
best_val, best_state, wait = 1e9, None, 0
for ep in range(1, MAX_EPOCHS+1):
    tl, ta = run_epoch(dl_train, True)
    vl, va = run_epoch(dl_val, False)
    print(f"epoch {ep:2d} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f}")
    if vl < best_val - 1e-4:
        best_val, best_state, wait = vl, copy.deepcopy(model.state_dict()), 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"ранняя остановка на эпохе {ep}"); break
model.load_state_dict(best_state)
print("загружены лучшие веса, val loss =", round(best_val,4))


In [ ]:
# Ячейка 7 — честная оценка на test: accuracy, per-class recall, confusion matrix
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt

model.eval(); y_true, y_pred = [], []
with torch.no_grad():
    for x, y in dl_test:
        out = model(x.to(DEVICE))
        y_pred += out.argmax(1).cpu().tolist(); y_true += y.tolist()

acc = accuracy_score(y_true, y_pred)
print("TEST accuracy:", round(acc, 4))
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5,4.4)); im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(3)); ax.set_xticklabels(CLASSES); ax.set_yticks(range(3)); ax.set_yticklabels(CLASSES)
ax.set_xlabel("предсказано"); ax.set_ylabel("истина"); ax.set_title(f"ResNet18 test acc={acc:.3f}")
for i in range(3):
    for j in range(3): ax.text(j,i,cm[i,j],ha="center",va="center",color="white" if cm[i,j]>cm.max()/2 else "black")
plt.tight_layout(); plt.savefig("confusion_matrix_cnn.png", dpi=120); plt.show()


In [ ]:
# Ячейка 8 — сохранить веса + метрики В GOOGLE DRIVE (надёжно) + печать метрик
import json, os, shutil
from sklearn.metrics import recall_score
metrics = {
    "model": "resnet18_finetune_layer4",
    "test_accuracy": round(float(acc), 4),
    "per_class_recall": {c: round(float(r), 4) for c, r in
        zip(CLASSES, recall_score(y_true, y_pred, average=None))},
    "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    "classes": CLASSES, "img_size": IMG_SIZE, "seed": SEED,
    "n_train": len(tr_files), "n_val": len(val_files), "n_test": len(test_files),
    "note": "Обучено на КУРИНОМ датасете. Высокая точность здесь ожидаема (артефакты источника). Реальная проверка — перепела, Этап 6."
}
# 1) сохраняем локально в Colab
with open("metrics.json", "w") as f: json.dump(metrics, f, ensure_ascii=False, indent=2)
torch.save(model.state_dict(), "verinbirds_resnet18.pt")

# 2) КОПИРУЕМ В GOOGLE DRIVE — файлы появятся в папке MyDrive/verinbirds_out/
OUT = "/content/drive/MyDrive/verinbirds_out"
os.makedirs(OUT, exist_ok=True)
for f in ["verinbirds_resnet18.pt", "metrics.json", "confusion_matrix_cnn.png"]:
    if os.path.exists(f): shutil.copy(f, OUT)
print("Файлы сохранены в Google Drive:", OUT)
print("  -> открой Drive, папку 'verinbirds_out', там 3 файла.\n")

# 3) метрики печатаем прямо здесь — можно просто скопировать из вывода в чат
print(json.dumps(metrics, ensure_ascii=False, indent=2))

# 4) авто-скачивание — как запасной вариант (может не сработать в Safari/Run all, это ок)
try:
    from google.colab import files
    for f in ["verinbirds_resnet18.pt", "metrics.json", "confusion_matrix_cnn.png"]:
        files.download(f)
except Exception as e:
    print("авто-скачивание не сработало (не страшно, файлы уже в Drive):", e)


## Готово

Файлы теперь лежат в **Google Drive → папка `verinbirds_out`** (надёжнее автоскачивания):
`verinbirds_resnet18.pt`, `metrics.json`, `confusion_matrix_cnn.png`.

Что сделать:
1. Скачай их из Drive и положи `verinbirds_resnet18.pt` в репозиторий: `ml/models/`.
2. **Сохрани напечатанный выше JSON с метриками для сравнения с baseline.**
   Нужны метрики (JSON), веса `.pt` — в `ml/models/`.
3. `confusion_matrix_cnn.png` можешь приложить, если хочешь.

Ещё способ забрать файлы: слева в Colab иконка папки (Files) → раскрой, найди файлы →
правой кнопкой → Download.

Напоминание: цель этапа — рабочие веса, а не «победить baseline на курах».
Кто лучше — решит перенос на перепелов (Этап 6).